In [ ]:
import unsloth
import torch
print(torch.__version__, "CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")

In [ ]:
!nvidia-smi

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv();
token = os.getenv("HUGGINGFACE_TOKEN")


login(token=token)


In [ ]:
from transformers import AutoTokenizer
model = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

print(model.vocab_size)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("allenai/ai2_arc", "ARC-Easy")["train"]

def schema_format(data):
    qid      = data["id"]
    question = data["question"]
    choices  = data["choices"]["text"]         
    labels   = data["choices"].get("label", None)
    answer   = data.get("answerKey")

    idx = labels.index(answer) if labels and answer in labels else ord(answer) - ord("A")   # If 'answer' is in 'labels', get 'idx'; else if just char, get 'idx'
    correct_choice = choices[idx]

    formatted_choices = "\n".join(f"{chr(ord('A')+i)}. {txt}" for i, txt in enumerate(choices))

    instruction = (
        f"ID: {qid}\n"
        f"Question: {question}\n"
        f"Choices:\n{formatted_choices}"
    )
    response = f"{answer}. {correct_choice}"

    if(model == 1): #HF
        return {"instruction": f"You are a careful MCQ solver.",
                "input": f"ID: {qid}\nQuestion: {question}\nChoices:\n{formatted_choices}\nAnswer:",
                "output": f"{response}"
                }
    elif(model == 2): #openAI
        return {
            "prompt": f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer:",
            "completion": f"{response}"
        }
    elif(model == 3): #Gemini
        return {
            "input_text": f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer:",
            "output_text": f"{response}"
        }
    elif(model == 4): #Claude
        return {
            "messages": [
                {"role": "user", "content": f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer:"},
                {"role": "assistant", "content": f"{response}"}
            ]
        }
    
dataset = dataset.map(schema_format)

model = 0
test_models = [1, 2, 3, 4]
import random
qn = random.randint(0, 2250)

for model_num in test_models:
    model = model_num
    test_dataset = dataset.map(schema_format)
    
    if(model_num == 1):
        print("\n=== HF ===")
        print(test_dataset[qn]["instruction"])
        print(test_dataset[qn]["input"])
        print(test_dataset[qn]["output"])
    elif(model_num == 2):
        print("\n=== OpenAI ===")
        print(test_dataset[qn]["prompt"])
        print(test_dataset[qn]["completion"])
    elif(model_num == 3):
        print("\n=== Gemini ===")
        print(test_dataset[qn]["input_text"])
        print(test_dataset[qn]["output_text"])
    elif(model_num == 4):
        print("\n=== Claude ===")
        print(test_dataset[qn]["messages"][0]["content"])
        print(test_dataset[qn]["messages"][1]["content"])